# AqSolDB | Aqueous Solubility Prediction

**Reproducible scientific report · Molecular machine learning / QSAR**

**Research objective:** Predict experimental aqueous solubility (LogS) from molecular SMILES.

**Final selected model:** Optuna-tuned XGBoost with 15 RDKit descriptors and 2,048-bit Morgan fingerprints.

**Recorded held-out performance:** R² **0.8044**, RMSE **1.0554**, MAE **0.6948**, on **1,013** test compounds.

> **Reproducibility boundary:** This notebook is a report of a *completed* experiment. It reads versioned data and stored results and displays diagnostics. It does **not** retrain models, repeat hyperparameter optimization, or conduct a second held-out test evaluation. Training and inference implementations are in `src/`, `scripts/`, and `main.nf`.

**Data and publication:** [AqSolDB dataset](https://doi.org/10.7910/DVN/OVHAW8) · [Sorkun, Khetan & Er (2019), *Scientific Data*](https://doi.org/10.1038/s41597-019-0151-1).

## 1 · Setup and provenance

Run this notebook with a Python environment containing the dependencies in `requirements.txt`. Open it from the repository root or `notebooks/` directory. All paths are resolved relative to the repository, not to a machine-specific upload folder.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Find the repository root from the current working directory.
ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "results" / "selected_model.json").is_file()
     and (p / "data" / "curated-solubility-dataset.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Cannot locate project files. Open the notebook in the AqSolDB repository."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

## 2 · Dataset and target distribution

The project uses the AqSolDB `curated-solubility-dataset.csv`: **9,982 original entries**. The finalized preprocessing removed two records with invalid SMILES, leaving **9,980** for modeling. The target is `Solubility` (LogS = log10 of molar aqueous solubility).

The original CSV includes some descriptor columns, but the final project **recomputes its 15 modeling descriptors from SMILES using RDKit**. Those source-supplied descriptor columns are not the model's feature-generation implementation.

In [ ]:
data = pd.read_csv(ROOT / "data" / "curated-solubility-dataset.csv")
print("Source records:", len(data))
print("Columns:", len(data.columns))
print("Missing SMILES:", data["SMILES"].isna().sum())
display(data[["ID", "SMILES", "Solubility"]].head())

In [ ]:
logs = pd.to_numeric(data["Solubility"], errors="coerce").dropna()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(logs, bins=50, edgecolor="white")
ax.set(xlabel="Experimental LogS", ylabel="Number of records",
       title="AqSolDB | Distribution of recorded aqueous solubilities")
fig.tight_layout()
plt.show()

## 3 · Chemistry-aware preprocessing and leakage control

- `src/preprocessing.py` canonicalizes valid SMILES with RDKit, retaining stereochemistry and disconnected chemical components. Invalid structures are rejected.
- `src/splitting.py` generates **molecular identity group keys**, stripping removable counterions and uncharging **only for grouping**. Model inputs are not silently neutralized or desalted.
- Two `GroupShuffleSplit` stages create approximately **80% training / 10% validation / 10% held-out test**, calculated over molecular groups rather than guaranteeing exact row percentages.
- All members of the same defined molecular group remain in one split. This protects against molecular-identity leakage, **not scaffold similarity leakage**. It is not an external- or scaffold-split benchmark.
- The selected model is fitted on **8,967 development records** (training + validation); the held-out partition contains **1,013 records**.

See `src/splitting.py`, `scripts/check_cv_splits.py`, and the saved result files for the underlying workflow.

## 4 · Feature representations

The project evaluates three RDKit-based representations:

| Representation | Definition | Features |
|:--|:--|--:|
| Descriptors | Physicochemical / structural descriptors | 15 |
| Fingerprints | Binary Morgan fingerprints, radius 2 | 2,048 |
| Hybrid | Descriptor array concatenated with fingerprint array | 2,063 |

Missing-value imputation and estimator-specific scaling are fitted inside the scikit-learn modeling pipelines—not globally across the dataset. The illustration below uses only three example molecules and **does not fit a model**.

In [ ]:
from src.preprocessing import canonicalize_smiles
from src.features import build_feature_matrices

examples = ["CCO", "c1ccccc1", "CC(=O)O"]
features = build_feature_matrices(
    [canonicalize_smiles(smiles) for smiles in examples]
)
for name, matrix in features.items():
    print(f"{name:12s} shape={matrix.shape}, dtype={matrix.dtype}")
assert features["hybrid"].shape == (3, 2063)

## 5 · Baseline comparison on development data

Five model families × three representations = **15 baseline configurations**: Ridge Regression, SVR, Random Forest, XGBoost, and LightGBM.

The CSV below records three-fold **group-aware cross-validation** results. These are not held-out test results. A separate initial validation leaderboard is preserved in `results/validation_leaderboard.csv`; the frozen selection was ultimately based on the group-CV/Optuna process, not on the test set.

In [ ]:
cv = pd.read_csv(ROOT / "results" / "cv_summary.csv")
cv = cv.sort_values("mean_r2", ascending=False).reset_index(drop=True)
assert len(cv) == 15
columns = ["model", "feature_set", "mean_r2", "std_r2", "mean_rmse", "mean_mae"]
display(cv[columns].round(4))

In [ ]:
top = cv.head(10).iloc[::-1]
labels = top["model"] + " | " + top["feature_set"]
fig, ax = plt.subplots(figsize=(9, 5.8))
ax.barh(labels, top["mean_r2"], xerr=top["std_r2"], capsize=2)
ax.set(xlim=(0, 1), xlabel="Mean group-aware 3-fold CV R²",
       title="Strongest baseline model–representation combinations")
fig.tight_layout()
plt.show()

## 6 · Frozen model selection

The strongest baseline candidates were **LightGBM + hybrid** (mean CV R² ≈ 0.8033) and **XGBoost + hybrid** (≈ 0.8014). Both were optimized using Optuna and three-fold group-aware CV on development data (25 trials per family).

The final choice was **Optuna-tuned XGBoost + hybrid**, selected at mean CV R² ≈ **0.8059**. Small CV differences among leading candidates are **not proof of statistically significant superiority**.

All selected hyperparameters, the frozen dataset SHA-256, and the selection provenance are stored in `results/selected_model.json`. The selection file predates the held-out evaluation.

In [ ]:
selection = json.loads(
    (ROOT / "results" / "selected_model.json").read_text(encoding="utf-8")
)
assert selection["model"] == "XGBoost"
assert selection["feature_set"] == "hybrid"
print("Selected model:", selection["model"])
print("Representation:", selection["feature_set"])
print("Selection method:", selection["cv_method"], selection["cv_folds"], "folds")
print(f"Selection CV mean R²: {selection['cv_mean_r2']:.4f}")
print("Optuna trial:", selection["optuna_trial"])
print("Dataset SHA-256:", selection["dataset_sha256"])
print("\nFrozen hyperparameters:")
print(json.dumps(selection["hyperparameters"], indent=2))

## 7 · One-time final held-out evaluation (recorded)

After selection was frozen, the chosen XGBoost pipeline was refitted on the **8,967 development compounds**. The **1,013 held-out compounds** were evaluated *once*. This notebook **reads the original metrics JSON** instead of producing a new evaluation.

- **Test R²:** 0.8044
- **Test RMSE:** 1.0554 LogS units
- **Test MAE:** 0.6948 LogS units

The 0.8059 selection cross-validation R² must not be conflated with the 0.8044 held-out test R².

In [ ]:
metrics = json.loads(
    (ROOT / "results" / "final_test_metrics.json").read_text(encoding="utf-8")
)
assert selection["dataset_sha256"] == metrics["dataset_sha256"]
assert metrics["n_train"] == 8967 and metrics["n_test"] == 1013

print(f"Development samples: {metrics['n_train']:,}")
print(f"Held-out test samples: {metrics['n_test']:,}")
print(f"Recorded test R²:   {metrics['test_r2']:.4f}")
print(f"Recorded test RMSE: {metrics['test_rmse']:.4f}")
print(f"Recorded test MAE:  {metrics['test_mae']:.4f}")

### Diagnostic plots from saved predictions

The following charts visualize `results/final_test_predictions.csv` **after the original completed evaluation**. They do not recompute performance metrics or make any new model-selection decisions.

In [ ]:
predictions = pd.read_csv(ROOT / "results" / "final_test_predictions.csv")
assert len(predictions) == metrics["n_test"]
observed = predictions["observed_logs"]
predicted = predictions["predicted_logs"]

lower = min(observed.min(), predicted.min())
upper = max(observed.max(), predicted.max())
fig, ax = plt.subplots(figsize=(6.2, 6.2))
ax.scatter(observed, predicted, alpha=0.35, s=14)
ax.plot([lower, upper], [lower, upper], linestyle="--", label="Perfect agreement")
ax.set(xlabel="Observed LogS", ylabel="Predicted LogS",
       title="Held-out predictions | Frozen XGBoost")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Residuals are read as recorded; this is a diagnostic visualization only.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(predictions["residual"].dropna(), bins=45, edgecolor="white")
ax.axvline(0, linestyle="--")
ax.set(xlabel="Recorded residual (observed − predicted), LogS",
       ylabel="Number of compounds", title="Held-out residual distribution")
fig.tight_layout()
plt.show()

## 8 · Run the reproducible Nextflow inference workflow

From the repository root in **Linux or WSL2**, with Docker and Nextflow installed:

```bash
# Build the environment (source + scripts are copied into the image).
docker build -t aqsoldb-solubility:dev .

# Create a small prediction dataset.
printf 'SMILES\nCCO\nc1ccccc1\nCC(=O)O\n' > molecules.csv

# Automatically reconstruct the frozen model and predict LogS.
nextflow run . --input_csv molecules.csv

# Reuse cached Nextflow stages on a subsequent run.
nextflow run . --input_csv molecules.csv -resume
```

The published output is `pipeline_outputs/nextflow/predictions.csv`.

When an existing trusted model bundle is available, add `--model_path /path/to/model.joblib` to skip reconstruction. This workflow does **not** repeat Optuna tuning or the held-out test evaluation.

Unit tests:

```bash
# Build the image first as above.
docker run --rm \
  --mount "type=bind,source=$(pwd)/tests,target=/app/tests,readonly" \
  aqsoldb-solubility:dev \
  python -m unittest discover -s tests -p 'test_*.py' -v
```

## 9 · Interpretation and limitations

The final recorded test R² of **0.8044** indicates good predictive fit for the project's held-out **molecular-identity-grouped** partition. It does **not** establish extrapolation to new chemical scaffolds, novel domains, or independent experimental sources.

**Important limitations:**

- **Splitting:** Normalized-identity group separation is not scaffold splitting; structurally similar distinct molecules can cross partitions.
- **Experimental variability:** AqSolDB integrates measurements from multiple sources and may contain heterogeneous experimental conditions and uncertainty.
- **Applicability domain:** No validated out-of-distribution detection or external-dataset assessment is reported.
- **Reproducibility:** Frozen dataset hash, split seed, and selected hyperparameters are retained, but most Python package versions are currently unpinned.
- **Usage:** These predictions are for research demonstration, not regulatory, clinical, or pharmaceutical decision-making.

**Source:** Sorkun, M. C., Khetan, A., & Er, S. (2019). *AqSolDB, a curated reference set of aqueous solubility and 2D descriptors for a diverse set of compounds*. *Scientific Data*, 6, 143. [DOI](https://doi.org/10.1038/s41597-019-0151-1). [Dataset](https://doi.org/10.7910/DVN/OVHAW8).